### **Import Libraries**

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.storagelevel import StorageLevel

### **Initialize Spark Session**

In [2]:
spark = (
    SparkSession.builder
    .appName("EEG_Schizophrenia_Feature_Engineering")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 07:46:09 WARN Utils: Your hostname, Sarras-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.234.219 instead (on interface en0)
26/04/13 07:46:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 07:46:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **Define Paths**

In [3]:
project_root = Path("..").resolve()

input_dir = project_root / "data" / "processed" / "all_csv_agg_ALLLL"
output_dir = project_root / "data" / "processed" / "feature_engineered"
output_dir.mkdir(parents=True, exist_ok=True)

final_output_dir = str(output_dir / "all_features_final")

### **Load the Reduced EEG Dataset**

In [4]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(input_dir))
)

### **Define Core Electrodes and ERP Windows**

In [5]:
electrodes = ["Fz", "FCz", "Cz", "FC3", "FC4", "C3", "C4", "CP3", "CP4"]
erp_windows = ["B0", "B1", "N100", "P200"]
stats_suffix = "avg"

regional_groups = {
    "frontal": ["Fz", "FCz"],
    "central": ["Cz", "C3", "C4"],
    "frontocentral": ["FC3", "FC4", "FCz"],
    "parietal": ["CP3", "CP4"],
    "left": ["FC3", "C3", "CP3"],
    "right": ["FC4", "C4", "CP4"],
    "global": electrodes,
}

asymmetry_pairs = {
    "fc": ("FC3", "FC4"),
    "c": ("C3", "C4"),
    "cp": ("CP3", "CP4"),
}

### **Define Reusable Feature Engineering Helpers**

In [6]:
def mean_expr(cols):
    return sum(F.col(c) for c in cols) / F.lit(len(cols))


def existing_cols(cols, df_columns):
    return [c for c in cols if c in df_columns]


def add_regional_features(df, groups, windows, stat_suffix):
    df_columns = set(df.columns)
    out = df

    for window in windows:
        for region_name, region_electrodes in groups.items():
            cols = [f"{elec}_{window}_{stat_suffix}" for elec in region_electrodes]
            cols = existing_cols(cols, df_columns)

            if cols:
                out = out.withColumn(f"{region_name}_{window}_{stat_suffix}", mean_expr(cols))

    return out


def add_baseline_adjusted_features(df, target_windows, baseline_window, stat_suffix):
    df_columns = set(df.columns)
    out = df

    for elec in electrodes:
        base_col = f"{elec}_{baseline_window}_{stat_suffix}"
        if base_col not in df_columns:
            continue

        for target_window in target_windows:
            target_col = f"{elec}_{target_window}_{stat_suffix}"
            if target_col in df_columns:
                out = out.withColumn(
                    f"{elec}_{target_window}_adj_{baseline_window}_{stat_suffix}",
                    F.col(target_col) - F.col(base_col)
                )

    for region_name in regional_groups.keys():
        base_col = f"{region_name}_{baseline_window}_{stat_suffix}"
        if base_col not in out.columns:
            continue

        for target_window in target_windows:
            target_col = f"{region_name}_{target_window}_{stat_suffix}"
            if target_col in out.columns:
                out = out.withColumn(
                    f"{region_name}_{target_window}_adj_{baseline_window}_{stat_suffix}",
                    F.col(target_col) - F.col(base_col)
                )

    return out


def add_n100_p200_relationship_features(df, stat_suffix):
    df_columns = set(df.columns)
    out = df

    for elec in electrodes:
        n100_col = f"{elec}_N100_{stat_suffix}"
        p200_col = f"{elec}_P200_{stat_suffix}"
        if n100_col in df_columns and p200_col in df_columns:
            out = out.withColumn(
                f"{elec}_N100_P200_diff_{stat_suffix}",
                F.col(n100_col) - F.col(p200_col)
            )

    for region_name in regional_groups.keys():
        n100_col = f"{region_name}_N100_{stat_suffix}"
        p200_col = f"{region_name}_P200_{stat_suffix}"
        if n100_col in out.columns and p200_col in out.columns:
            out = out.withColumn(
                f"{region_name}_N100_P200_diff_{stat_suffix}",
                F.col(n100_col) - F.col(p200_col)
            )

    return out


def add_asymmetry_features(df, windows, stat_suffix):
    df_columns = set(df.columns)
    out = df

    for window in windows:
        for pair_name, (left_elec, right_elec) in asymmetry_pairs.items():
            left_col = f"{left_elec}_{window}_{stat_suffix}"
            right_col = f"{right_elec}_{window}_{stat_suffix}"

            if left_col in df_columns and right_col in df_columns:
                out = out.withColumn(
                    f"{pair_name}_{window}_asym_{stat_suffix}",
                    F.col(left_col) - F.col(right_col)
                )

    if (
        f"left_N100_{stat_suffix}" in out.columns
        and f"right_N100_{stat_suffix}" in out.columns
    ):
        out = out.withColumn(
            f"hemisphere_N100_asym_{stat_suffix}",
            F.col(f"left_N100_{stat_suffix}") - F.col(f"right_N100_{stat_suffix}")
        )

    if (
        f"left_P200_{stat_suffix}" in out.columns
        and f"right_P200_{stat_suffix}" in out.columns
    ):
        out = out.withColumn(
            f"hemisphere_P200_asym_{stat_suffix}",
            F.col(f"left_P200_{stat_suffix}") - F.col(f"right_P200_{stat_suffix}")
        )

    if (
        f"left_B0_{stat_suffix}" in out.columns
        and f"right_B0_{stat_suffix}" in out.columns
    ):
        out = out.withColumn(
            f"hemisphere_B0_asym_{stat_suffix}",
            F.col(f"left_B0_{stat_suffix}") - F.col(f"right_B0_{stat_suffix}")
        )

    if (
        f"left_B1_{stat_suffix}" in out.columns
        and f"right_B1_{stat_suffix}" in out.columns
    ):
        out = out.withColumn(
            f"hemisphere_B1_asym_{stat_suffix}",
            F.col(f"left_B1_{stat_suffix}") - F.col(f"right_B1_{stat_suffix}")
        )

    return out

### **Build Trial-Level Engineered Features**

In [7]:
trial_features_df = df

trial_features_df = add_regional_features(
    df=trial_features_df,
    groups=regional_groups,
    windows=erp_windows,
    stat_suffix=stats_suffix,
)

trial_features_df = add_baseline_adjusted_features(
    df=trial_features_df,
    target_windows=["N100", "P200"],
    baseline_window="B0",
    stat_suffix=stats_suffix,
)

trial_features_df = add_n100_p200_relationship_features(
    df=trial_features_df,
    stat_suffix=stats_suffix,
)

trial_features_df = add_asymmetry_features(
    df=trial_features_df,
    windows=erp_windows,
    stat_suffix=stats_suffix,
)

### **Define Condition Mapping for Cross-Condition Features**

In [8]:
active_condition = 1
passive_condition = 2
control_condition = 3

### **Compute Subject-Level Mean ERP Features by Condition**

In [9]:
subject_condition_agg_exprs = []

for elec in electrodes:
    for window in ["N100", "P200", "B0", "B1"]:
        col_name = f"{elec}_{window}_{stats_suffix}"
        if col_name in trial_features_df.columns:
            subject_condition_agg_exprs.append(
                F.avg(F.col(col_name)).alias(col_name)
            )

subject_condition_means_df = (
    trial_features_df
    .groupBy("subject", "condition")
    .agg(*subject_condition_agg_exprs)
)

### **Create N100 Suppression Features Across Conditions**

In [10]:
suppression_join_base = (
    subject_condition_means_df
    .select(
        "subject",
        "condition",
        *[c for c in subject_condition_means_df.columns if c.endswith("_N100_avg")]
    )
)

active_df = (
    suppression_join_base
    .filter(F.col("condition") == active_condition)
    .drop("condition")
)

passive_df = (
    suppression_join_base
    .filter(F.col("condition") == passive_condition)
    .drop("condition")
)

suppression_df = active_df.alias("a").join(
    passive_df.alias("p"),
    on="subject",
    how="inner"
)

for elec in electrodes:
    passive_col = f"p.{elec}_N100_avg"
    active_col = f"a.{elec}_N100_avg"
    output_col = f"{elec}_N100_supp_avg"

    suppression_df = suppression_df.withColumn(
        output_col,
        F.col(passive_col) - F.col(active_col)
    )

suppression_select_cols = ["subject"] + [f"{elec}_N100_supp_avg" for elec in electrodes]

suppression_df = suppression_df.select(*suppression_select_cols)

### **Create Regional and Global Suppression Features**

In [11]:
suppression_df = add_regional_features(
    df=suppression_df,
    groups=regional_groups,
    windows=["N100_supp"],
    stat_suffix="avg",
)

# The function expects columns like region_window_suffix.
# Since suppression columns are in the form electrode_N100_supp_avg,
# the above creates region_N100_supp_avg correctly.

### **Compute Trial-to-Trial Variability Features**

In [12]:
variability_target_cols = []
for elec in electrodes:
    col_name = f"{elec}_N100_{stats_suffix}"
    if col_name in trial_features_df.columns:
        variability_target_cols.append(col_name)

variability_exprs = []
for col_name in variability_target_cols:
    variability_exprs.extend([
        F.stddev(F.col(col_name)).alias(f"{col_name}_trial_std"),
        (F.max(F.col(col_name)) - F.min(F.col(col_name))).alias(f"{col_name}_trial_range"),
        (
            F.stddev(F.col(col_name)) / (F.abs(F.avg(F.col(col_name))) + F.lit(1e-6))
        ).alias(f"{col_name}_trial_cv")
    ])

subject_variability_df = (
    trial_features_df
    .groupBy("subject", "condition")
    .agg(*variability_exprs)
)

### **Aggregate Variability to Subject Level**

In [13]:
subject_variability_agg_exprs = []

for col_name in subject_variability_df.columns:
    if col_name not in {"subject", "condition"}:
        subject_variability_agg_exprs.append(
            F.avg(F.col(col_name)).alias(col_name)
        )

subject_variability_final_df = (
    subject_variability_df
    .groupBy("subject")
    .agg(*subject_variability_agg_exprs)
)

### **Build the Final Subject-Level Feature Table**

In [14]:
subject_features_df = (
    suppression_df
    .join(subject_variability_final_df, on="subject", how="outer")
)

### **Join Subject-Level Features Back to Trial-Level Rows**

In [15]:
final_features_df = (
    trial_features_df
    .join(subject_features_df, on="subject", how="left")
)

### **Build a Reduced Final Modeling Table**

In [16]:
electrodes = ["Fz", "FCz", "Cz", "FC3", "FC4", "C3", "C4", "CP3", "CP4"]
windows = ["B0", "B1", "N100", "P200"]

id_cols = ["subject", "trial", "condition"]

raw_keep_cols = [
    f"{elec}_{window}_avg"
    for elec in electrodes
    for window in windows
    if f"{elec}_{window}_avg" in final_features_df.columns
]

engineered_prefixes = (
    "frontal_",
    "central_",
    "frontocentral_",
    "parietal_",
    "left_",
    "right_",
    "global_",
)

engineered_contains = (
    "_adj_B0_",
    "_N100_P200_diff_",
    "_asym_",
    "_supp_",
    "_trial_std",
    "_trial_range",
    "_trial_cv",
)

engineered_cols = [
    c for c in final_features_df.columns
    if (
        c.startswith(engineered_prefixes)
        or any(token in c for token in engineered_contains)
    )
]

final_modeling_cols = []
seen = set()

for c in id_cols + raw_keep_cols + engineered_cols:
    if c in final_features_df.columns and c not in seen:
        final_modeling_cols.append(c)
        seen.add(c)

final_modeling_df = final_features_df.select(*final_modeling_cols)

### **Validate the Reduced Modeling Table**

In [17]:
final_modeling_df.select(
    "subject",
    "trial",
    "condition"
).show(5, truncate=False)

+-------+-----+---------+
|subject|trial|condition|
+-------+-----+---------+
|1.0    |1.0  |1.0      |
|1.0    |1.0  |2.0      |
|1.0    |2.0  |1.0      |
|1.0    |2.0  |2.0      |
|1.0    |2.0  |3.0      |
+-------+-----+---------+
only showing top 5 rows


In [18]:
final_modeling_df.select(
    "subject",
    "trial",
    "condition",
    "global_N100_avg",
    "global_B0_avg"
).show(5, truncate=False)

+-------+-----+---------+-------------------+---------------------+
|subject|trial|condition|global_N100_avg    |global_B0_avg        |
+-------+-----+---------+-------------------+---------------------+
|1.0    |1.0  |1.0      |-10.145080952380955|0.04545652642934187  |
|1.0    |1.0  |2.0      |-16.859412169312165|-0.023995577130528703|
|1.0    |2.0  |1.0      |6.68533544973545   |0.06234056094929942  |
|1.0    |2.0  |2.0      |-0.87731746031746  |0.021083171521035893 |
|1.0    |2.0  |3.0      |8.04295873015873   |0.06845706580366767  |
+-------+-----+---------+-------------------+---------------------+
only showing top 5 rows


### **Save the Reduced Final Table**

In [19]:
reduced_output_dir = str(output_dir / "final_modeling_table")

(
    final_modeling_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(reduced_output_dir)
)

26/04/13 07:48:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### **Stop Spark Session**

In [20]:
spark.stop()